# Train Growth Rate

In [ ]:
from utils import *
from torch.utils.data import DataLoader
import h5py, numpy as np
from tqdm import tqdm
RAW_DIR = 'raw_ds' if LOCAL else '/nfsd/automatica/grandinmat/raw_ds' # where the raw data is stored

In [ ]:
# read the h5 file k1: 'shot' -> ['Tz', 'ei', 't']
h5_file_path = 'growth_rate_db.h5'
h5_file = h5py.File(h5_file_path, 'r')
print(h5_file.keys())

# open key '79060'
key = '79060'
data = h5_file[key]
print(data.keys())
t = np.array(data['t']).reshape(-1)
# round t to 5 decimal places
t = np.round(t, 4)
print(t[:20])

In [ ]:
# iterate 
phys, fxs, tzs, eis = [], [], [], []
for key in tqdm(h5_file.keys()):
    d1 = h5_file[key]
    t1 = np.round(np.array(d1['t']).reshape(-1), 4)
    nt1 = len(t1)
    ei = np.array(d1['ei']).reshape(1, nt1)
    tz = np.array(d1['Tz']).reshape(19, nt1)

    # load the mat file
    mat_file_path = f'{RAW_DIR}/{key}.mat'
    assert os.path.exists(mat_file_path), f'Mat file {mat_file_path} does not exist'
    d2 = loadmat(mat_file_path)

    ph = np.concatenate([d2[k] for k in INPUT_NAMES], axis=0)
    fx = d2[FX]
    t2 = np.round(d2['t'].reshape(-1), 4)

    assert np.all(np.isin(t1, t2)), f'Not all t1 are in t2 for key {key}'

    # keep only the indices in t2 that are in t1
    indices = np.where(np.isin(t2, t1))[0]
    tt = t2[indices]
    assert np.all(tt == t1), f't1 and tt do not match for key {key}' 

    if len(indices) > 1:
        phys.append(ph[:, indices])
        fxs.append(fx[:,:,indices])
        tzs.append(tz)
        eis.append(ei)

phys = np.concatenate(phys, axis=1)
fxs = np.concatenate(fxs, axis=2)
tzs = np.concatenate(tzs, axis=1)
eis = np.concatenate(eis, axis=1)

print(f'phys: {phys.shape}, fxs: {fxs.shape}, tzs: {tzs.shape}, eis: {eis.shape}')


In [ ]:
# create dataset
class GRds(Dataset):
    def __init__(self, x, tzs, eis): self.x, self.tzs, self.eis = to_tensor(x, DEV), to_tensor(tzs, DEV), to_tensor(eis, DEV)
    def __len__(self): return self.x.shape[1]
    def __getitem__(self, idx): return (self.x[:, idx], self.tzs[:, idx], self.eis[:, idx])

ds = GRds(phys, tzs, eis)
print(f'Dataset length: {len(ds)}')

dl = DataLoader(ds, batch_size=64, shuffle=True)

In [ ]:
# network
lnn = load_net_from_data(data_dir='data/3535303', model='best_Fx.pth')

class GRNet(Module):
    def __init__(self):
        super(GRNet, self).__init__()
        self.emb2gr = Sequential(
            Linear(PHYSICS_LS, 32), ActF(32),
            Linear(32, 19+1), ActF(19+1)
        )
    def forward(self, x):
        gr_out = self.emb2gr(x)
        return gr_out[:, :19], gr_out[:, 19:]  # tz, ei

In [ ]:
# calculate all the embeddings from phys
embs = torch.zeros((PHYSICS_LS, len(ds)), device=DEV)
with torch.no_grad():
    for i in tqdm(range(len(ds))):
        x, _, _ = ds[i]
        x = x.unsqueeze(0)  # add batch dimension
        x = (x - lnn.x_mean_std[0]) / lnn.x_mean_std[1]
        emb = lnn.phys_enc(x)
        embs[:, i] = emb.squeeze(0)
eds = GRds(embs, tzs, eis)